# Notebook 02 v2 — NSL-KDD Model Training

**Methodology change vs v1:** trained on the 80% train slice from Notebook 01 v2 (100,778 samples), not the full official training set (125,973 samples). The remaining 20% (25,195 samples) is reserved for calibrator fitting in Notebook 03 v2.

**Models trained (9 total):**

Canonical 6 (used by all downstream notebooks):
- `rf_binary_cw`, `xgb_binary_cw`, `dnn_binary_cw` — binary classification, class-weighted
- `rf_5class_smote`, `xgb_5class_smote`, `dnn_5class_smote` — 5-class, SMOTE oversampling

Ablation 3 (justifies SMOTE choice for 5-class):
- `rf_5class_cw`, `xgb_5class_cw`, `dnn_5class_cw` — 5-class, class-weighted (alternative to SMOTE)

**Hyperparameters identical to v1** — only the training data changes. This keeps the v1→v2 comparison methodologically clean (any metric shift is attributable to the smaller training set, not model definition changes).

**Outputs written to `models/nsl_kdd_v2/`:**

| File | Notes |
|---|---|
| `{name}.pkl` or `{name}.pt` | Trained model weights |
| `predictions/{name}_test_pred.npy` | Predictions on official test set |
| `predictions/{name}_test_proba.npy` | Probabilities on official test set |
| `predictions/{name}_calib_pred.npy` | **NEW** — predictions on calibration holdout |
| `predictions/{name}_calib_proba.npy` | **NEW** — probabilities on calibration holdout (input to Notebook 03 v2) |
| `metrics.json` | All 9 model metrics |
| `imbalance_ablation.csv` | Paper-ready CW-vs-SMOTE comparison on 5-class |

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json, pickle, time
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU detected. Switch to T4 runtime: Runtime → Change runtime type → T4 GPU.')

Device: cuda
CUDA: Tesla T4


In [3]:
# v2 paths
PROCESSED = Path(REPO) / 'data' / 'processed' / 'nsl_kdd_v2'
MODELS_DIR = Path(REPO) / 'models' / 'nsl_kdd_v2'
PREDS_DIR = MODELS_DIR / 'predictions'
TABLES_DIR = Path(REPO) / 'results' / 'tables'
for d in [MODELS_DIR, PREDS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Load v2 data: train (80% slice), calib (20% slice), test (official, untouched)
X_train = np.load(PROCESSED / 'X_train.npy')
X_calib = np.load(PROCESSED / 'X_calib.npy')
X_test  = np.load(PROCESSED / 'X_test.npy')

y_train_b = np.load(PROCESSED / 'y_train_binary.npy')
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_test_b  = np.load(PROCESSED / 'y_test_binary.npy')

y_train_5 = np.load(PROCESSED / 'y_train_5class.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')
y_test_5  = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]
CLASS_NAMES_BIN = ['Normal', 'Attack']

print(f'Train: {X_train.shape}  (80% slice)')
print(f'Calib: {X_calib.shape}  (20% slice — fed to Notebook 03 v2)')
print(f'Test:  {X_test.shape}   (untouched official test)')
print(f'\nTrain 5-class distribution:')
for i in range(5):
    print(f'  {CLASS_NAMES_5[i]:8s}: {np.sum(y_train_5==i):>6,}')

Train: (100778, 122)  (80% slice)
Calib: (25195, 122)  (20% slice — fed to Notebook 03 v2)
Test:  (22544, 122)   (untouched official test)

Train 5-class distribution:
  Normal  : 53,874
  DoS     : 36,741
  Probe   :  9,325
  R2L     :    796
  U2R     :     42


## 2. Hyperparameters and evaluation helpers

In [4]:
# Hyperparameters (identical to v1)
RF_HP = dict(n_estimators=200, n_jobs=-1, random_state=SEED)
XGB_HP = dict(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9, tree_method='hist',
    random_state=SEED, n_jobs=-1, eval_metric='mlogloss',
)
DNN_HP = dict(
    hidden=(256, 128, 64, 32), dropout=0.3,
    batch_size=256, lr=1e-3, weight_decay=1e-5, epochs=30,
)

# DNN architecture
class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden, dropout):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def evaluate(y_true, y_pred, labels):
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0, labels=labels)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
        'f1_per_class': [float(f) for f in f1_per_class],
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }

def save_predictions(name, X_test_arr, X_calib_arr, predict_fn):
    """Save predictions+probas on both test and calibration sets."""
    test_pred, test_proba = predict_fn(X_test_arr)
    calib_pred, calib_proba = predict_fn(X_calib_arr)
    np.save(PREDS_DIR / f'{name}_test_pred.npy',  test_pred)
    np.save(PREDS_DIR / f'{name}_test_proba.npy', test_proba)
    np.save(PREDS_DIR / f'{name}_calib_pred.npy',  calib_pred)
    np.save(PREDS_DIR / f'{name}_calib_proba.npy', calib_proba)
    return test_pred, test_proba

ALL_METRICS = {}
print('Helpers and hyperparameters ready.')

Helpers and hyperparameters ready.


## 3. RF, XGB, DNN training functions

In [5]:
def train_rf(X_tr, y_tr, n_classes, name, class_weight=None):
    print(f'\n--- {name} ---')
    t0 = time.time()
    model = RandomForestClassifier(**RF_HP, class_weight=class_weight)
    model.fit(X_tr, y_tr)
    print(f'  Trained in {time.time()-t0:.1f}s')

    with open(MODELS_DIR / f'{name}.pkl', 'wb') as f:
        pickle.dump(model, f)

    def predict_fn(X):
        return model.predict(X), model.predict_proba(X)

    return model, predict_fn

def train_xgb(X_tr, y_tr, n_classes, name, sample_weight=None):
    print(f'\n--- {name} ---')
    t0 = time.time()
    hp = dict(XGB_HP)
    if n_classes == 2:
        hp['objective'] = 'binary:logistic'
    else:
        hp['objective'] = 'multi:softprob'
        hp['num_class'] = n_classes
    model = xgb.XGBClassifier(**hp)
    model.fit(X_tr, y_tr, sample_weight=sample_weight)
    print(f'  Trained in {time.time()-t0:.1f}s')

    with open(MODELS_DIR / f'{name}.pkl', 'wb') as f:
        pickle.dump(model, f)

    def predict_fn(X):
        return model.predict(X), model.predict_proba(X)

    return model, predict_fn

def train_dnn(X_tr, y_tr, n_classes, name, class_weights=None):
    print(f'\n--- {name} ---')
    t0 = time.time()
    hp = DNN_HP
    model = MLP(X_tr.shape[1], n_classes, hp['hidden'], hp['dropout']).to(DEVICE)

    X_t = torch.tensor(X_tr, dtype=torch.float32)
    y_t = torch.tensor(y_tr, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=hp['batch_size'], shuffle=True)

    if class_weights is not None:
        cw = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=cw)
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])

    model.train()
    for epoch in range(1, hp['epochs']+1):
        ep_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * xb.size(0)
        ep_loss /= len(loader.dataset)
        if epoch == 1 or epoch % 5 == 0 or epoch == hp['epochs']:
            print(f'  Epoch {epoch:>2}  loss={ep_loss:.4f}')
    print(f'  Trained in {time.time()-t0:.1f}s')

    torch.save({
        'state_dict': model.state_dict(),
        'in_dim': X_tr.shape[1],
        'n_classes': n_classes,
        'hidden': list(hp['hidden']),
        'dropout': hp['dropout'],
    }, MODELS_DIR / f'{name}.pt')

    def predict_fn(X):
        model.eval()
        with torch.no_grad():
            X_te = torch.tensor(X, dtype=torch.float32).to(DEVICE)
            logits = model(X_te)
            proba = torch.softmax(logits, dim=1).cpu().numpy()
            pred = proba.argmax(axis=1)
        return pred, proba

    return model, predict_fn

print('Training functions ready.')

Training functions ready.


## 4. Train binary models (class-weighted)

In [6]:
labels_b = [0, 1]
cw_b = compute_class_weight('balanced', classes=np.array(labels_b), y=y_train_b)
cw_b_dict = {i: cw_b[i] for i in labels_b}
sample_w_b = np.array([cw_b[v] for v in y_train_b])

print(f'Binary class weights: {dict(zip(labels_b, cw_b))}')

# RF binary CW
_, predict_fn = train_rf(X_train, y_train_b, 2, 'rf_binary_cw', class_weight=cw_b_dict)
test_pred, _ = save_predictions('rf_binary_cw', X_test, X_calib, predict_fn)
ALL_METRICS['rf_binary_cw'] = evaluate(y_test_b, test_pred, labels_b)
print(f'  Test: acc={ALL_METRICS["rf_binary_cw"]["accuracy"]:.4f}  f1m={ALL_METRICS["rf_binary_cw"]["f1_macro"]:.4f}  mcc={ALL_METRICS["rf_binary_cw"]["mcc"]:.4f}')

# XGB binary CW
_, predict_fn = train_xgb(X_train, y_train_b, 2, 'xgb_binary_cw', sample_weight=sample_w_b)
test_pred, _ = save_predictions('xgb_binary_cw', X_test, X_calib, predict_fn)
ALL_METRICS['xgb_binary_cw'] = evaluate(y_test_b, test_pred, labels_b)
print(f'  Test: acc={ALL_METRICS["xgb_binary_cw"]["accuracy"]:.4f}  f1m={ALL_METRICS["xgb_binary_cw"]["f1_macro"]:.4f}  mcc={ALL_METRICS["xgb_binary_cw"]["mcc"]:.4f}')

# DNN binary CW
_, predict_fn = train_dnn(X_train, y_train_b, 2, 'dnn_binary_cw', class_weights=cw_b)
test_pred, _ = save_predictions('dnn_binary_cw', X_test, X_calib, predict_fn)
ALL_METRICS['dnn_binary_cw'] = evaluate(y_test_b, test_pred, labels_b)
print(f'  Test: acc={ALL_METRICS["dnn_binary_cw"]["accuracy"]:.4f}  f1m={ALL_METRICS["dnn_binary_cw"]["f1_macro"]:.4f}  mcc={ALL_METRICS["dnn_binary_cw"]["mcc"]:.4f}')

Binary class weights: {0: np.float64(0.9353120243531202), 1: np.float64(1.0743006993006994)}

--- rf_binary_cw ---
  Trained in 15.1s
  Test: acc=0.7674  f1m=0.7663  mcc=0.6031

--- xgb_binary_cw ---
  Trained in 12.3s
  Test: acc=0.7883  f1m=0.7878  mcc=0.6331

--- dnn_binary_cw ---
  Epoch  1  loss=0.0895
  Epoch  5  loss=0.0243
  Epoch 10  loss=0.0200
  Epoch 15  loss=0.0180
  Epoch 20  loss=0.0159
  Epoch 25  loss=0.0142
  Epoch 30  loss=0.0132
  Trained in 64.1s
  Test: acc=0.8106  f1m=0.8105  mcc=0.6652


## 5. Train 5-class SMOTE models (canonical)

In [7]:
# SMOTE on the v2 train slice. k_neighbors handled if any class is too small.
min_class_size = int(np.min(np.bincount(y_train_5)))
k_neighbors = min(5, min_class_size - 1)
print(f'Train min class size: {min_class_size}, using SMOTE k_neighbors={k_neighbors}')

smote = SMOTE(random_state=SEED, k_neighbors=k_neighbors)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train_5)

print(f'\nAfter SMOTE oversampling:')
for i in range(5):
    print(f'  {CLASS_NAMES_5[i]:8s}: pre={np.sum(y_train_5==i):>6,}  post={np.sum(y_train_sm==i):>6,}')
print(f'Total: {len(X_train):,} → {len(X_train_sm):,}')

Train min class size: 42, using SMOTE k_neighbors=5

After SMOTE oversampling:
  Normal  : pre=53,874  post=53,874
  DoS     : pre=36,741  post=53,874
  Probe   : pre= 9,325  post=53,874
  R2L     : pre=   796  post=53,874
  U2R     : pre=    42  post=53,874
Total: 100,778 → 269,370


In [8]:
labels_5 = list(range(5))

# RF 5-class SMOTE
_, predict_fn = train_rf(X_train_sm, y_train_sm, 5, 'rf_5class_smote')
test_pred, _ = save_predictions('rf_5class_smote', X_test, X_calib, predict_fn)
ALL_METRICS['rf_5class_smote'] = evaluate(y_test_5, test_pred, labels_5)
m = ALL_METRICS['rf_5class_smote']
print(f'  Test: acc={m["accuracy"]:.4f}  f1m={m["f1_macro"]:.4f}  mcc={m["mcc"]:.4f}  | R2L={m["f1_per_class"][3]:.3f}  U2R={m["f1_per_class"][4]:.3f}')

# XGB 5-class SMOTE
_, predict_fn = train_xgb(X_train_sm, y_train_sm, 5, 'xgb_5class_smote')
test_pred, _ = save_predictions('xgb_5class_smote', X_test, X_calib, predict_fn)
ALL_METRICS['xgb_5class_smote'] = evaluate(y_test_5, test_pred, labels_5)
m = ALL_METRICS['xgb_5class_smote']
print(f'  Test: acc={m["accuracy"]:.4f}  f1m={m["f1_macro"]:.4f}  mcc={m["mcc"]:.4f}  | R2L={m["f1_per_class"][3]:.3f}  U2R={m["f1_per_class"][4]:.3f}')

# DNN 5-class SMOTE
_, predict_fn = train_dnn(X_train_sm, y_train_sm, 5, 'dnn_5class_smote')
test_pred, _ = save_predictions('dnn_5class_smote', X_test, X_calib, predict_fn)
ALL_METRICS['dnn_5class_smote'] = evaluate(y_test_5, test_pred, labels_5)
m = ALL_METRICS['dnn_5class_smote']
print(f'  Test: acc={m["accuracy"]:.4f}  f1m={m["f1_macro"]:.4f}  mcc={m["mcc"]:.4f}  | R2L={m["f1_per_class"][3]:.3f}  U2R={m["f1_per_class"][4]:.3f}')


--- rf_5class_smote ---
  Trained in 70.4s
  Test: acc=0.7427  f1m=0.5102  mcc=0.6297  | R2L=0.031  U2R=0.154

--- xgb_5class_smote ---
  Trained in 132.0s
  Test: acc=0.7869  f1m=0.6138  mcc=0.6920  | R2L=0.187  U2R=0.413

--- dnn_5class_smote ---
  Epoch  1  loss=0.1877
  Epoch  5  loss=0.0471
  Epoch 10  loss=0.0375
  Epoch 15  loss=0.0328
  Epoch 20  loss=0.0314
  Epoch 25  loss=0.0298
  Epoch 30  loss=0.0286
  Trained in 153.9s
  Test: acc=0.7812  f1m=0.5747  mcc=0.6795  | R2L=0.194  U2R=0.243


## 6. Train 5-class class-weighted ablation variants

These three models train on the un-resampled 5-class data with `class_weight='balanced'` (or equivalent for XGB and DNN). They are *ablation* variants — included so the paper can justify the SMOTE choice over class-weighting on 5-class. Used only in the appendix imbalance-strategy comparison; not used by any downstream stage.

In [9]:
cw_5 = compute_class_weight('balanced', classes=np.array(labels_5), y=y_train_5)
cw_5_dict = {i: cw_5[i] for i in labels_5}
sample_w_5 = np.array([cw_5[v] for v in y_train_5])

print(f'5-class class weights:')
for i in range(5):
    print(f'  {CLASS_NAMES_5[i]:8s}: {cw_5[i]:.4f}')

# RF 5-class CW
_, predict_fn = train_rf(X_train, y_train_5, 5, 'rf_5class_cw', class_weight=cw_5_dict)
test_pred, _ = save_predictions('rf_5class_cw', X_test, X_calib, predict_fn)
ALL_METRICS['rf_5class_cw'] = evaluate(y_test_5, test_pred, labels_5)
m = ALL_METRICS['rf_5class_cw']
print(f'  Test: acc={m["accuracy"]:.4f}  f1m={m["f1_macro"]:.4f}  mcc={m["mcc"]:.4f}  | R2L={m["f1_per_class"][3]:.3f}  U2R={m["f1_per_class"][4]:.3f}')

# XGB 5-class CW
_, predict_fn = train_xgb(X_train, y_train_5, 5, 'xgb_5class_cw', sample_weight=sample_w_5)
test_pred, _ = save_predictions('xgb_5class_cw', X_test, X_calib, predict_fn)
ALL_METRICS['xgb_5class_cw'] = evaluate(y_test_5, test_pred, labels_5)
m = ALL_METRICS['xgb_5class_cw']
print(f'  Test: acc={m["accuracy"]:.4f}  f1m={m["f1_macro"]:.4f}  mcc={m["mcc"]:.4f}  | R2L={m["f1_per_class"][3]:.3f}  U2R={m["f1_per_class"][4]:.3f}')

# DNN 5-class CW
_, predict_fn = train_dnn(X_train, y_train_5, 5, 'dnn_5class_cw', class_weights=cw_5)
test_pred, _ = save_predictions('dnn_5class_cw', X_test, X_calib, predict_fn)
ALL_METRICS['dnn_5class_cw'] = evaluate(y_test_5, test_pred, labels_5)
m = ALL_METRICS['dnn_5class_cw']
print(f'  Test: acc={m["accuracy"]:.4f}  f1m={m["f1_macro"]:.4f}  mcc={m["mcc"]:.4f}  | R2L={m["f1_per_class"][3]:.3f}  U2R={m["f1_per_class"][4]:.3f}')

5-class class weights:
  Normal  : 0.3741
  DoS     : 0.5486
  Probe   : 2.1615
  R2L     : 25.3211
  U2R     : 479.8952

--- rf_5class_cw ---
  Trained in 15.0s
  Test: acc=0.7345  f1m=0.4691  mcc=0.6173  | R2L=0.007  U2R=0.029

--- xgb_5class_cw ---
  Trained in 45.1s
  Test: acc=0.7842  f1m=0.5948  mcc=0.6883  | R2L=0.132  U2R=0.366

--- dnn_5class_cw ---
  Epoch  1  loss=0.6004
  Epoch  5  loss=0.1898
  Epoch 10  loss=0.1351
  Epoch 15  loss=0.1215
  Epoch 20  loss=0.1251
  Epoch 25  loss=0.0947
  Epoch 30  loss=0.0974
  Trained in 60.7s
  Test: acc=0.7716  f1m=0.5471  mcc=0.6637  | R2L=0.219  U2R=0.093


## 7. Summary table

In [10]:
rows = []
for name, m in ALL_METRICS.items():
    rows.append({
        'Model': name,
        'Accuracy': m['accuracy'],
        'F1 macro': m['f1_macro'],
        'F1 weighted': m['f1_weighted'],
        'MCC': m['mcc'],
        'R2L F1': m['f1_per_class'][3] if len(m['f1_per_class']) == 5 else None,
        'U2R F1': m['f1_per_class'][4] if len(m['f1_per_class']) == 5 else None,
    })
df = pd.DataFrame(rows)
print('ALL 9 MODELS — NSL-KDD v2')
print('=' * 100)
print(df.to_string(index=False, float_format='%.4f'))
print('=' * 100)
df.to_csv(TABLES_DIR / 'nslkdd_v2_model_comparison.csv', index=False)
print(f'\nSaved: {TABLES_DIR / "nslkdd_v2_model_comparison.csv"}')

ALL 9 MODELS — NSL-KDD v2
           Model  Accuracy  F1 macro  F1 weighted    MCC  R2L F1  U2R F1
    rf_binary_cw    0.7674    0.7663       0.7640 0.6031     NaN     NaN
   xgb_binary_cw    0.7883    0.7878       0.7863 0.6331     NaN     NaN
   dnn_binary_cw    0.8106    0.8105       0.8099 0.6652     NaN     NaN
 rf_5class_smote    0.7427    0.5102       0.6974 0.6297  0.0313  0.1538
xgb_5class_smote    0.7869    0.6138       0.7525 0.6920  0.1870  0.4130
dnn_5class_smote    0.7812    0.5747       0.7501 0.6795  0.1941  0.2432
    rf_5class_cw    0.7345    0.4691       0.6866 0.6173  0.0069  0.0286
   xgb_5class_cw    0.7842    0.5948       0.7455 0.6883  0.1318  0.3656
   dnn_5class_cw    0.7716    0.5471       0.7514 0.6637  0.2190  0.0935

Saved: /content/drive/MyDrive/XIDS_Research/xids-research/results/tables/nslkdd_v2_model_comparison.csv


## 8. Paper-ready imbalance-strategy ablation table

Compares CW vs SMOTE per architecture on 5-class. Justifies the SMOTE choice for the canonical pipeline.

In [11]:
ablation_rows = []
for arch in ['rf', 'xgb', 'dnn']:
    for strat in ['cw', 'smote']:
        name = f'{arch}_5class_{strat}'
        m = ALL_METRICS[name]
        ablation_rows.append({
            'Architecture': arch.upper(),
            'Strategy': 'class-weighted' if strat == 'cw' else 'SMOTE',
            'Accuracy': m['accuracy'],
            'Macro F1': m['f1_macro'],
            'MCC': m['mcc'],
            'R2L F1': m['f1_per_class'][3],
            'U2R F1': m['f1_per_class'][4],
        })
ablation_df = pd.DataFrame(ablation_rows)
print('5-CLASS IMBALANCE STRATEGY ABLATION — NSL-KDD v2')
print('=' * 100)
print(ablation_df.to_string(index=False, float_format='%.4f'))
print('=' * 100)
ablation_df.to_csv(TABLES_DIR / 'nslkdd_v2_imbalance_ablation.csv', index=False)
print(f'\nSaved: {TABLES_DIR / "nslkdd_v2_imbalance_ablation.csv"}')

5-CLASS IMBALANCE STRATEGY ABLATION — NSL-KDD v2
Architecture       Strategy  Accuracy  Macro F1    MCC  R2L F1  U2R F1
          RF class-weighted    0.7345    0.4691 0.6173  0.0069  0.0286
          RF          SMOTE    0.7427    0.5102 0.6297  0.0313  0.1538
         XGB class-weighted    0.7842    0.5948 0.6883  0.1318  0.3656
         XGB          SMOTE    0.7869    0.6138 0.6920  0.1870  0.4130
         DNN class-weighted    0.7716    0.5471 0.6637  0.2190  0.0935
         DNN          SMOTE    0.7812    0.5747 0.6795  0.1941  0.2432

Saved: /content/drive/MyDrive/XIDS_Research/xids-research/results/tables/nslkdd_v2_imbalance_ablation.csv


## 9. Save metrics.json and commit

In [12]:
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(ALL_METRICS, f, indent=2)

print(f'✓ Saved {len(ALL_METRICS)} model metrics to: {MODELS_DIR / "metrics.json"}')
print(f'\nModels on disk:')
for p in sorted(MODELS_DIR.iterdir()):
    if p.is_file() and p.suffix in ['.pkl', '.pt']:
        size_kb = p.stat().st_size / 1024
        print(f'  {p.name:<35}  {size_kb:>10,.1f} KB')

print(f'\nPredictions saved:')
n_pred_files = sum(1 for p in PREDS_DIR.iterdir() if p.is_file())
print(f'  {n_pred_files} files in {PREDS_DIR}')
print(f'  Per model: 4 files (test_pred, test_proba, calib_pred, calib_proba)')
print(f'  Expected total: 9 models × 4 = 36 files')

✓ Saved 9 model metrics to: /content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/metrics.json

Models on disk:
  dnn_5class_cw.pt                          310.2 KB
  dnn_5class_smote.pt                       310.3 KB
  dnn_binary_cw.pt                          309.8 KB
  rf_5class_cw.pkl                       28,436.6 KB
  rf_5class_smote.pkl                    35,575.8 KB
  rf_binary_cw.pkl                       18,541.1 KB
  xgb_5class_cw.pkl                       2,823.9 KB
  xgb_5class_smote.pkl                    3,379.2 KB
  xgb_binary_cw.pkl                         803.1 KB

Predictions saved:
  36 files in /content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/predictions
  Per model: 4 files (test_pred, test_proba, calib_pred, calib_proba)
  Expected total: 9 models × 4 = 36 files


In [13]:
os.chdir(REPO)
!git add notebooks/02_train_models_v2.ipynb
!git add results/tables/nslkdd_v2_model_comparison.csv
!git add results/tables/nslkdd_v2_imbalance_ablation.csv
!git status --short
!git commit -m 'Notebook 02 v2: NSL-KDD model training on 80% train slice (9 models: 6 canonical + 3 imbalance ablation)'
!git push origin main

Refresh index: 100% (227/227), done.
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
AM notebooks/02_train_models_v2.ipynb
A  results/tables/nslkdd_v2_imbalance_ablation.csv
A  results/tables/nslkdd_v2_model_comparison.csv
?? calibrators/
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? results/figures/unsw_dnn_5class_diagnostic.png
?? shap_values/unsw_nb15/
[main aecaad2] Notebook 02 v2: NSL-KDD model training on 80% train slice (9 models: 6 canonical + 3 imbalance ablation)
 3 files changed, 18 insertions(+)
 create mode 100644 notebooks/02_train_models_v2.ipynb
 create mode 100644 results/tables/nslkdd_v2_imbalance_ablation.csv
 create mode 100644 results/tables/nslkdd_v2_model_comparison.csv
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 8.48 KiB | 868.00 KiB/s

In [14]:
import os
from pathlib import Path

REPO = Path('/content/drive/MyDrive/XIDS_Research/xids-research')
PREDS = REPO / 'models' / 'nsl_kdd_v2' / 'predictions'

print(f'Files in {PREDS}:')
if PREDS.exists():
    files = sorted(PREDS.iterdir())
    print(f'  Total: {len(files)} files (expected 36 = 9 models × 4 files)')
    for f in files[:8]:  # show first 8 as a sample
        print(f'  {f.name}  ({f.stat().st_size:,} bytes)')
    if len(files) > 8:
        print(f'  ... and {len(files)-8} more')
else:
    print('  Directory does not exist — training may have failed')

# Also check metrics.json
metrics_path = REPO / 'models' / 'nsl_kdd_v2' / 'metrics.json'
if metrics_path.exists():
    import json
    with open(metrics_path) as f:
        m = json.load(f)
    print(f'\n✓ metrics.json saved with {len(m)} models: {list(m.keys())}')
else:
    print('\n✗ metrics.json missing')

Files in /content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/predictions:
  Total: 36 files (expected 36 = 9 models × 4 files)
  dnn_5class_cw_calib_pred.npy  (201,688 bytes)
  dnn_5class_cw_calib_proba.npy  (504,028 bytes)
  dnn_5class_cw_test_pred.npy  (180,480 bytes)
  dnn_5class_cw_test_proba.npy  (451,008 bytes)
  dnn_5class_smote_calib_pred.npy  (201,688 bytes)
  dnn_5class_smote_calib_proba.npy  (504,028 bytes)
  dnn_5class_smote_test_pred.npy  (180,480 bytes)
  dnn_5class_smote_test_proba.npy  (451,008 bytes)
  ... and 28 more

✓ metrics.json saved with 9 models: ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw', 'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote', 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']
